# NNDL Assignment 4 - GCN + Transformer (Kaggle GPU runner)

**Before running:** attach the `nndl_project4` dataset (Add Input), then in Settings set
**Accelerator = GPU** and **Internet = On** (requires a phone-verified account).

Run the cells in order. Cell 5 is a fast smoke test that must pass before the full run in cell 7.


## 1. Confirm the GPU is attached


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Install extra packages (torch is already on Kaggle)


In [ ]:
!pip install -q torch_geometric rdkit selfies


## 3. Copy the project into the writable working dir


In [ ]:
import os, shutil
src = None
for root, _, files in os.walk('/kaggle/input'):
    if 'main.py' in files and 'config.py' in files:
        src = root; break
assert src, 'Project not found under /kaggle/input - is the dataset attached?'
proj = '/kaggle/working/nndl_project4'
if os.path.exists(proj): shutil.rmtree(proj)
shutil.copytree(src, proj)
os.chdir(proj)
print('Running from:', proj)


## 4. SMOKE TEST (about 1-2 minutes)

Tiny subset, 1 warm-up epoch, 2 joint epochs, 40 samples per temperature.
This only checks that the pipeline runs end to end and writes every artifact.
Its numbers are meaningless and its outputs are deleted in cell 6.


In [ ]:
!cd /kaggle/working/nndl_project4 && SMOKE=1 python main.py 2>&1 | tail -40


## 5. Verify the smoke run produced everything


In [ ]:
import json, os
base = '/kaggle/working/nndl_project4/output/section2'
req = ['results.json', 'rejected_examples.json', 'checkpoint.pt', 'training_log.txt',
       'lm_pretrain_curves.png', 'joint_training_curves.png',
       'generation_metrics.png', 'accepted_novel_grid.png']
missing = [f for f in req if not os.path.exists(os.path.join(base, f))]
print('MISSING FILES:', missing if missing else 'none')
r = json.load(open(os.path.join(base, 'results.json')))
print('lm_history present      :', 'lm_history' in r)
print('joint_history present   :', 'joint_history' in r)
print('random_chance_top_10    :', r['retrieval'].get('random_chance_top_10'))
print('eos_fallback_count      :', r.get('eos_fallback_count'))
rej = json.load(open(os.path.join(base, 'rejected_examples.json')))
print('rejected examples/temp  :', {t: len(v) for t, v in rej.items()})
nn = r['embedding_nn_examples']
print('nn examples             :', len(nn))
print('nn keys                 :', sorted(nn[0].keys()) if nn else 'NONE')
assert not missing, 'smoke run did not produce all artifacts - stop and report this'
print()
print('SMOKE TEST PASSED - safe to run the full pipeline')


## 6. Clear the smoke outputs, then run the FULL pipeline

Full settings from `config.py`: 15 warm-up epochs, up to 40 joint epochs,
batch 256, 300 samples at each of T = 0.8, 1.0, 1.2.


In [ ]:
import shutil, os
shutil.rmtree('/kaggle/working/nndl_project4/output', ignore_errors=True)
os.makedirs('/kaggle/working/nndl_project4/output/section1', exist_ok=True)
os.makedirs('/kaggle/working/nndl_project4/output/section2', exist_ok=True)
print('smoke outputs cleared')


In [ ]:
!cd /kaggle/working/nndl_project4 && python main.py 2>&1 | tee /kaggle/working/full_run_console.txt


## 7. Verification summary - paste this whole output back


In [ ]:
import json
base = '/kaggle/working/nndl_project4/output/section2'
r = json.load(open(base + '/results.json'))
rej = json.load(open(base + '/rejected_examples.json'))

print('=== TRAINING ===')
lm, jt = r['lm_history'], r['joint_history']
print('LM  val_ppl first/last :', round(lm['val_ppl'][0], 3), round(lm['val_ppl'][-1], 3))
print('LM  val_acc first/last :', round(lm['val_acc'][0], 4), round(lm['val_acc'][-1], 4))
print('joint epochs run       :', len(jt['train_total']))
print('joint con  first/last  :', round(jt['train_contrastive'][0], 4), round(jt['train_contrastive'][-1], 4))
print('joint pair first/last  :', round(jt['train_pair'][0], 4), round(jt['train_pair'][-1], 4))
print('joint lm   first/last  :', round(jt['train_lm'][0], 4), round(jt['train_lm'][-1], 4))
print('joint ret_acc last     :', round(jt['val_retrieval_acc'][-1], 4))
print('joint val_ppl first/last:', round(jt['val_ppl'][0], 3), round(jt['val_ppl'][-1], 3))

print()
print('=== RETRIEVAL ===')
print(json.dumps(r['retrieval'], indent=1))

print()
print('=== GENERATION ===')
for t, rep in r['generation'].items():
    print('T =', t)
    print('  counts :', rep['counts'])
    print('  metrics:', {k: round(v, 4) for k, v in rep['metrics'].items()})

print()
print('=== REJECTED EXAMPLES ===')
for t, items in rej.items():
    print('T =', t)
    for e in items:
        print('   [%s] %s' % (e['stage'], e['smiles']))

print()
print('=== NEAREST-NEIGHBOUR EXAMPLES ===')
print('eos_fallback_count:', r.get('eos_fallback_count'))
for e in r['embedding_nn_examples']:
    print(' gen  =', e['smiles'])
    print('   T=%s gs=%.4f nn=%.4f fp=%.4f fallback=%s'
          % (e['temperature'], e['graph_sequence_similarity'],
             e['nearest_embedding_similarity'], e['max_fingerprint_similarity'],
             e['eos_fallback_used']))
    print('   near =', e['nearest_training_molecule'])

print()
print('unique accepted novel:', r['n_accepted_novel_unique'])
print('dataset stats        :', r['dataset_stats'])


## 8. Zip everything for download


In [ ]:
import shutil
shutil.copy('/kaggle/working/full_run_console.txt',
            '/kaggle/working/nndl_project4/output/section2/console_output.txt')
shutil.make_archive('/kaggle/working/nndl_outputs', 'zip',
                    '/kaggle/working/nndl_project4/output')
print('Download nndl_outputs.zip from the right-side Output panel.')
